# Fit-stratified sensitivity analysis

This notebook uses only the four PSO output files already in `data/model_outputs/`
(`PSO_2018/2019/2020/2021_6params_NYC_norm_28_PSO_15.csv`). Each file has a per-CBG `cost`
column, which is `1 - Pearson r` from the PSO calibration objective. We define
`fit = 1 - cost` and use it to test whether the main pandemic-shock and non-reversion
findings are driven by the CBGs where the model fits worst.

Outputs are written to `outputs/item2_fit_sensitivity/` and correspond to Table 1
(fit distribution), Table 6 (fit-stratified % change), and Supplementary Tables S7-S8
(quartile breakdowns) in the revised manuscript.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import os

YEARS = [2018, 2019, 2020, 2021]
DATA_DIR = "../../data/model_outputs"
OUT_DIR = "../../outputs/item2_fit_sensitivity"
os.makedirs(OUT_DIR, exist_ok=True)

PARAM_MAP = {
    'H_Area_of_store': 'Store area',
    'R_Percentage_of_Visits_by_brand': 'Chain loyalty',
    'J_POI_count_where_store_is': 'POI count',
    'K_POI_diversity_where_store_is': 'POI diversity',
    'L_Demographic_similarity': 'Demographic similarity',
    'G_Distance_between_cbg_and_store': 'CBG-Store Distance'
}


## Load PSO output files and compute per-CBG fit (1 - cost)

In [ ]:
dfs = {}
for y in YEARS:
    path = f"{DATA_DIR}/PSO_{y}_6params_NYC_norm_28_PSO_15.csv"
    df = pd.read_csv(path)
    df['cbg'] = df['cbg'].astype(str)
    df = df.set_index('cbg')
    df['fit'] = 1 - df['cost']
    dfs[y] = df
    print(y, df.shape)

common = sorted(set.intersection(*[set(dfs[y].index) for y in YEARS]))
print("Common CBGs across all 4 years:", len(common))
assert len(common) == 5502, "Expected 5,502 CBGs in the analytical sample"


## Table 1 (main manuscript): CBG-level fit distribution by year

In [ ]:
rows = []
for y in YEARS:
    f = dfs[y].loc[common, 'fit']
    rows.append({
        'year': y, 'mean': f.mean(), 'median': f.median(),
        'p10': f.quantile(0.10), 'p25': f.quantile(0.25),
        'p75': f.quantile(0.75), 'p90': f.quantile(0.90),
    })
fit_dist = pd.DataFrame(rows)
print(fit_dist.round(3).to_string(index=False))
fit_dist.round(4).to_csv(f"{OUT_DIR}/table_A_fit_distribution_by_year.csv", index=False)


## Helper: percentage change in mean parameter value between two years, over a given set of CBGs

In [ ]:
def pct_change(y1, y2, idx):
    d1, d2 = dfs[y1], dfs[y2]
    out = {}
    for col, name in PARAM_MAP.items():
        m1 = d1.loc[idx, col].mean()
        m2 = d2.loc[idx, col].mean()
        out[name] = (m2 - m1) / m1 * 100
    return out

def fit_filtered_idx(focal_year, exclude_pct, base_idx=common):
    f = dfs[focal_year].loc[base_idx, 'fit']
    thresh = f.quantile(exclude_pct)
    return f[f > thresh].index.tolist()


## Table 6 (main manuscript): % change excluding the bottom 10/20/25% of CBGs by fit

In [ ]:
results = {}
for (y1, y2, focal) in [(2019, 2020, 2020), (2019, 2021, 2021), (2020, 2021, 2021)]:
    row = {'Full sample (n=5502)': pct_change(y1, y2, common)}
    for exclude_pct, label in [(0.10, 'Exclude bottom 10% fit'),
                                 (0.20, 'Exclude bottom 20% fit'),
                                 (0.25, 'Exclude bottom 25% fit')]:
        idx = fit_filtered_idx(focal, exclude_pct)
        row[f'{label} (n={len(idx)})'] = pct_change(y1, y2, idx)
    key = f'{y1}-{y2}'
    results[key] = row
    df_out = pd.DataFrame(row).T
    print("="*90); print(f"{y1} to {y2} (filtered on {focal} fit)")
    print(df_out.round(2).to_string())
    df_out.round(2).to_csv(f"{OUT_DIR}/table_B_pct_change_{y1}_{y2}_fit_filtered.csv")


## Supplementary Tables S7-S8: fit-quartile breakdown 
 Split CBGs into quartiles of a given year's fit and repeat the comparison within each quartile — the more stringent test reported alongside Table 6.

In [ ]:
def quartile_pct_change(y1, y2, quartile_year):
    fit_q = dfs[quartile_year].loc[common, 'fit']
    quartile_labels = pd.qcut(fit_q, 4, labels=['Q1 (lowest fit)', 'Q2', 'Q3', 'Q4 (highest fit)'])
    quartile_map = pd.Series(quartile_labels, index=fit_q.index)
    rows = {}
    for q in ['Q1 (lowest fit)', 'Q2', 'Q3', 'Q4 (highest fit)']:
        idx = quartile_map[quartile_map == q].index.tolist()
        rows[f'{q} (n={len(idx)})'] = pct_change(y1, y2, idx)
    return pd.DataFrame(rows).T

# Table S7: 2019 -> 2020, quartiles of 2020 fit
q_2020 = quartile_pct_change(2019, 2020, quartile_year=2020)
print("2019 -> 2020 by 2020-fit quartile:")
print(q_2020.round(2).to_string())
q_2020.round(2).to_csv(f"{OUT_DIR}/table_C_quartile_2019_2020.csv")

# Table S8: 2019 -> 2021, quartiles of 2021 fit
q_2021 = quartile_pct_change(2019, 2021, quartile_year=2021)
print("\n2019 -> 2021 by 2021-fit quartile:")
print(q_2021.round(2).to_string())
q_2021.round(2).to_csv(f"{OUT_DIR}/table_C_quartile_2019_2021.csv")


## Summary 
 Direction of the main effects (store area up, chain loyalty up, distance down) is unchanged after excluding the poorest-fitting CBGs and is consistent across all four fit quartiles, including the lowest. See Section 5.3.4 and Table 6 of the revised manuscript, and Supplementary Tables S6-S8, for the full write-up and interpretation.